In [1]:
import sys
sys.path.append("/kaggle/input/library-metrics")

In [2]:
import sys, os, time, threading
sys.path.append("/kaggle/input/library-metrics")

import numpy as np
import pandas as pd
import psutil, tracemalloc
from sklearn.feature_selection import SelectKBest, chi2
from metric_toolkit import (
    compute_energy, compute_carbon, compute_edp
)

# ============================================================
# 1️⃣ DỮ LIỆU
# ============================================================
X_train = pd.read_csv('/kaggle/input/data-time-complexity/X_train.csv')
y_train = pd.read_csv('/kaggle/input/data-time-complexity/y_train.csv').values.ravel()

# ============================================================
# 2️⃣ HÀM FEATURE SELECTION
# ============================================================
def run_chi_square(k=100):
    selector = SelectKBest(score_func=chi2, k=min(k, X_train.shape[1]))
    X_sel = selector.fit_transform(X_train, y_train)
    mask = selector.get_support()
    num_evals = X_train.shape[1]
    return mask, num_evals, X_sel

# ============================================================
# 3️⃣ LOGGER BỘ NHỚ KIỂU TRACEMALLOC
# ============================================================
mem_trace_log = []
logging_active = False

def log_tracemalloc(interval=0.5):
    start_time = time.time()
    while logging_active:
        current, peak = tracemalloc.get_traced_memory()
        t = time.time() - start_time
        mem_trace_log.append((t, current / 1024**2, peak / 1024**2))  # MB
        time.sleep(interval)

# ============================================================
# 4️⃣ ĐO TOÀN BỘ CHỈ SỐ
# ============================================================
def measure_time_and_memory(func, *args, **kwargs):
    proc = psutil.Process(os.getpid())
    cpu_readings = []
    monitoring = True

    def monitor_cpu():
        while monitoring:
            cpu_readings.append(psutil.cpu_percent(interval=0.5))

    cpu_thread = threading.Thread(target=monitor_cpu)
    cpu_thread.start()

    tracemalloc.start()
    start_cpu = time.process_time()
    start_time = time.time()

    # Bắt đầu logger bộ nhớ
    global logging_active
    logging_active = True
    logger_thread = threading.Thread(target=log_tracemalloc, args=(0.1,))
    logger_thread.start()

    result = func(*args, **kwargs)

    # Dừng logger và CPU monitor
    logging_active = False
    logger_thread.join()
    monitoring = False
    cpu_thread.join()

    end_time = time.time()
    end_cpu = time.process_time()
    current_mem, peak_mem = tracemalloc.get_traced_memory()
    tracemalloc.stop()

    wall = end_time - start_time
    cpu_time = end_cpu - start_cpu
    avg_cpu = np.mean(cpu_readings) if cpu_readings else 0
    max_cpu = np.max(cpu_readings) if cpu_readings else 0
    peak_mem_mb = peak_mem / (1024 * 1024)

    return {
        "result": result,
        "WallTime(s)": wall,
        "CPUTime(s)": cpu_time,
        "CPUUtil_Avg(%)": avg_cpu,
        "CPUUtil_Max(%)": max_cpu,
        "PeakMem(MB)": peak_mem_mb,
    }

# ============================================================
# 5️⃣ CHẠY FS + ĐO CHỈ SỐ
# ============================================================
timing = measure_time_and_memory(run_chi_square)
mask, num_evals, X_sel = timing["result"]

wall_time = timing["WallTime(s)"]
cpu_util = timing["CPUUtil_Avg(%)"]
peak_mem = timing["PeakMem(MB)"]
base_mem = psutil.Process().memory_info().rss / (1024*1024)

energy = compute_energy(cpu_util, wall_time)
carbon = compute_carbon(energy)
edp = compute_edp(energy, wall_time)

# ============================================================
# 6️⃣ LƯU FILE CSV
# ============================================================
fs_metrics = {
    "FS_Method": "ChiSquare",
    "NumFeatures": int(mask.sum()),
    "NumEvals": num_evals,
    "WallTime(s)": wall_time,
    "CPUUtil(%)": cpu_util,
    "PeakMem(MB)": peak_mem,
    "BaseMem(MB)": base_mem,
    "Energy(J)": energy,
    "Carbon(gCO2e)": carbon,
    "EDP(J*s)": edp,
    "Timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
}

# Lưu kết quả chính
np.save("/kaggle/working/chi_mask.npy", mask)
pd.DataFrame([fs_metrics]).to_csv("/kaggle/working/fs_metrics_chi.csv", index=False)

# Lưu log bộ nhớ kiểu tracemalloc
mem_df = pd.DataFrame(mem_trace_log, columns=["Time(s)", "Current(MB)", "Peak(MB)"])
mem_df.to_csv("/kaggle/working/tracemalloc_timeline_chi.csv", index=False)

# ============================================================
# 7️⃣ IN KẾT QUẢ
# ============================================================
print("Chi-Squared FS done.")
print(f"→ Selected features: {mask.sum()}")
print(f"→ WallTime: {wall_time:.3f}s | CPU Avg: {cpu_util:.2f}%")
print(f"→ PeakMem (Python heap): {peak_mem:.2f} MB")
print(f"→ Energy: {energy:.2f} J | Carbon: {carbon:.4f} gCO2e")
print(f"→ Saved: fs_metrics_chi.csv, tracemalloc_timeline_chi.csv, chi_mask.npy")


Chi-Squared FS done.
→ Selected features: 100
→ WallTime: 0.501s | CPU Avg: 15.00%
→ PeakMem (Python heap): 122.57 MB
→ Energy: 1.98 J | Carbon: 0.0003 gCO2e
→ Saved: fs_metrics_chi.csv, tracemalloc_timeline_chi.csv, chi_mask.npy
